In [2]:
import wntr

In [3]:
from wntr.epanet.util import to_si, FlowUnits, HydParam

path = r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\ky9.inp"
wn = wntr.network.WaterNetworkModel(path)

EMITTER_C_TO_SI = to_si(FlowUnits.GPM, 1.0, HydParam.EmitterCoeff)
GPM_TO_M3S = 0.0000630902
FT_TO_M = 0.3048

### Getting detils of the pipeline network

In [4]:
test_wn = wntr.network.WaterNetworkModel(path)

print("Nodes:", test_wn.node_name_list)
print("Links:", test_wn.link_name_list)
print("Junctions:", test_wn.junction_name_list)
print("Reservoirs:", test_wn.reservoir_name_list)
print("Pipes:", test_wn.pipe_name_list)
print("Valves:", test_wn.valve_name_list)
print("Tanks:", test_wn.tank_name_list)
print("Pumps:", test_wn.pump_name_list)

print("\nPipe details:")
for name in test_wn.pipe_name_list:
    p = test_wn.get_link(name)
    print(name, ":", p.start_node_name, "->", p.end_node_name,
          "| L =", p.length, "| D =", p.diameter, "| C =", p.roughness)

Nodes: ['J-1', 'J-10', 'J-100', 'J-1001', 'J-1002', 'J-1003', 'J-1004', 'J-1005', 'J-1006', 'J-1007', 'J-1008', 'J-101', 'J-1010', 'J-1011', 'J-1012', 'J-1013', 'J-1014', 'J-1015', 'J-1016', 'J-1017', 'J-1018', 'J-1019', 'J-102', 'J-1020', 'J-1021', 'J-1022', 'J-1023', 'J-1024', 'J-1025', 'J-1026', 'J-1027', 'J-1028', 'J-1029', 'J-103', 'J-1030', 'J-1031', 'J-1032', 'J-1033', 'J-1034', 'J-1035', 'J-1036', 'J-1037', 'J-1038', 'J-1039', 'J-104', 'J-1041', 'J-1042', 'J-1044', 'J-1045', 'J-1046', 'J-1047', 'J-1048', 'J-1049', 'J-105', 'J-1050', 'J-1051', 'J-1052', 'J-1053', 'J-1054', 'J-1055', 'J-1056', 'J-1057', 'J-1058', 'J-1059', 'J-106', 'J-1060', 'J-1061', 'J-1062', 'J-1064', 'J-1065', 'J-1066', 'J-1067', 'J-1068', 'J-1069', 'J-107', 'J-1070', 'J-1071', 'J-1072', 'J-1073', 'J-1074', 'J-1075', 'J-1077', 'J-1078', 'J-1079', 'J-108', 'J-1081', 'J-1082', 'J-1083', 'J-1084', 'J-1085', 'J-1086', 'J-1087', 'J-1089', 'J-109', 'J-1090', 'J-1091', 'J-1092', 'J-1093', 'J-1094', 'J-1095', 'J-1096

The network consists of 1096 junctions, 1270 pipes, 56 valves, 15 tanks, 17 pumps, 4 reservoirs


### Inspecting EPANET and WNTR unit/convention

In [5]:
# ---------------------------------------------------------
# 1. EPANET conventions used by THIS network
# ---------------------------------------------------------
hyd = wn.options.hydraulic

print("=" * 70)
print("EPANET CONVENTIONS — CURRENT NETWORK")
print("=" * 70)

print("INP flow units       :", hyd.inpfile_units)
print("INP pressure units   :", hyd.inpfile_pressure_units)
print("Headloss formula     :", hyd.headloss)
print("Demand model         :", hyd.demand_model)
print("Emitter exponent     :", hyd.emitter_exponent)
print("Specific gravity     :", hyd.specific_gravity)

# ---------------------------------------------------------
# 2. All EPANET-supported flow units
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("EPANET FLOW UNITS")
print("=" * 70)

for unit in FlowUnits:
    print(
        f"{unit.name:6} | "
        f"traditional={unit.is_traditional} | "
        f"metric={unit.is_metric} | "
        f"SI factor={unit.factor}"
    )

# ---------------------------------------------------------
# 3. EPANET parameter categories handled by WNTR
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("EPANET PARAMETERS CONVERTED BY WNTR")
print("=" * 70)

for param in HydParam:
    print(f"{param.name:20} | {param}")

EPANET CONVENTIONS — CURRENT NETWORK
INP flow units       : GPM
INP pressure units   : None
Headloss formula     : H-W
Demand model         : DDA
Emitter exponent     : 0.5
Specific gravity     : 1.0

EPANET FLOW UNITS
CFS    | traditional=True | metric=False | SI factor=0.0283168466
GPM    | traditional=True | metric=False | SI factor=6.30901964e-05
MGD    | traditional=True | metric=False | SI factor=0.043812636388888895
IMGD   | traditional=True | metric=False | SI factor=0.05261678240740741
AFD    | traditional=True | metric=False | SI factor=0.014276410185185185
LPS    | traditional=False | metric=True | SI factor=0.001
LPM    | traditional=False | metric=True | SI factor=1.6666666666666667e-05
MLD    | traditional=False | metric=True | SI factor=0.011574074074074073
CMH    | traditional=False | metric=True | SI factor=0.0002777777777777778
CMD    | traditional=False | metric=True | SI factor=1.1574074074074073e-05
SI     | traditional=False | metric=False | SI factor=1.0

EPANET 

### Experimental Setup

In [ ]:
import copy
import os
import tempfile
import numpy as np

# ---------------------------------------------------------
# IMPORTANT:
# wn must already contain the realistic network.
# ---------------------------------------------------------

# Immutable baseline for all one-variable-at-a-time tests
base_wn = copy.deepcopy(wn)

# Pick representative existing components
J_TEST = base_wn.junction_name_list[0]
R_TEST = base_wn.reservoir_name_list[0]
P_TEST = base_wn.pipe_name_list[0]

T_TEST = base_wn.tank_name_list[0] if base_wn.tank_name_list else None
PU_TEST = base_wn.pump_name_list[0] if base_wn.pump_name_list else None
V_TEST = base_wn.valve_name_list[0] if base_wn.valve_name_list else None

print("Test junction :", J_TEST)
print("Test reservoir:", R_TEST)
print("Test pipe     :", P_TEST)
print("Test tank     :", T_TEST)
print("Test pump     :", PU_TEST)
print("Test valve    :", V_TEST)

Test junction : J-1
Test reservoir: R-1
Test pipe     : P-1
Test tank     : T-1
Test pump     : ~@Pump-1
Test valve    : ~@RV-1


##### Hydraulic Simulation Helper

In [7]:
def run_hydraulics(model, hours=24):
    """
    Runs EPANET hydraulics using the current in-memory model.

    Temporary engine files are created in a temporary directory
    and deleted afterwards. The original .inp file is never written.
    """
    model.options.time.duration = int(hours * 3600)
    model.options.time.hydraulic_timestep = 3600
    model.options.time.pattern_timestep = 3600
    model.options.time.report_timestep = 3600

    with tempfile.TemporaryDirectory() as tmpdir:
        results = wntr.sim.EpanetSimulator(model).run_sim(
            file_prefix=os.path.join(tmpdir, "run"),
            convergence_error=True
        )

    return results

`EpanetSimulator` runs the EPANET hydraulic engine; simulation duration and timesteps are properties of the in-memory model. [EPA GitHub Pages](https://usepa.github.io/WNTR/apidoc/wntr.sim.epanet.EpanetSimulator.html)

### 1. Baseline hydraulic simulation

First make sure the realistic network itself works.

In [8]:
wn = copy.deepcopy(base_wn)

baseline_results = run_hydraulics(wn)

pressure = baseline_results.node["pressure"]

print("Baseline simulation: SUCCESS")
print("Minimum pressure:", pressure.min().min())
print("Maximum pressure:", pressure.max().max())

Baseline simulation: SUCCESS
Minimum pressure: -28.273495
Maximum pressure: 272.11023


### 2. Global demand magnitude

You already changed one junction's demand manually. This is the more useful new experiment: **scale the demand of the entire network**.


In [9]:
wn = copy.deepcopy(base_wn)

original = wn.options.hydraulic.demand_multiplier

for multiplier in [0.8, 1.0, 1.2]:

    wn.options.hydraulic.demand_multiplier = multiplier

    print("\nDemand multiplier:", wn.options.hydraulic.demand_multiplier)

    assert np.isclose(
        wn.options.hydraulic.demand_multiplier,
        multiplier
    )

    results = run_hydraulics(wn)

    mean_pressure = results.node["pressure"].iloc[0].mean()

    print("Verified: YES")
    print("Mean pressure:", mean_pressure)


Demand multiplier: 0.8
Verified: YES
Mean pressure: 63.51531

Demand multiplier: 1.0
Verified: YES
Mean pressure: 63.345814

Demand multiplier: 1.2
Verified: YES
Mean pressure: 63.1712


`demand_multiplier` scales the baseline demands of the junctions. [EPA GitHub Pages](https://usepa.github.io/WNTR/apidoc/wntr.network.options.HydraulicOptions.html)

### 3. Realistic 24-hour demand pattern

You already tested a short `[0.8, 1.0, 1.2]` pattern. Now use a proper daily pattern while keeping its average approximately equal to the original demand.

In [10]:
wn = copy.deepcopy(base_wn)

pattern_values = np.array([
    0.70, 0.65, 0.60, 0.60, 0.70, 0.90,
    1.20, 1.30, 1.15, 1.00, 0.95, 0.90,
    0.95, 1.00, 1.05, 1.10, 1.20, 1.30,
    1.25, 1.15, 1.05, 0.95, 0.85, 0.75
])

# Normalize so average multiplier = 1
pattern_values = pattern_values / pattern_values.mean()

pattern_name = "Experimental_24h_Demand"

wn.add_pattern(pattern_name, pattern_values)

junction = wn.get_node(J_TEST)
junction.demand_timeseries_list[0].pattern_name = pattern_name

print("Junction:", J_TEST)
print("Pattern:", junction.demand_timeseries_list[0].pattern_name)
print("Average multiplier:", np.mean(pattern_values))

assert junction.demand_timeseries_list[0].pattern_name == pattern_name
assert np.isclose(np.mean(pattern_values), 1.0)

print("Verified: YES")

Junction: J-1
Pattern: Experimental_24h_Demand
Average multiplier: 1.0
Verified: YES


A demand pattern is a sequence of multipliers applied over time; the pattern timestep determines when the multipliers change. [EPA GitHub Pages](https://usepa.github.io/WNTR/apidoc/wntr.network.elements.Pattern.html)

In [ ]:
results = run_hydraulics(wn, hours=24)

print("Simulation: SUCCESS")
print(
    "Pressure at test junction:",
    results.node["pressure"][J_TEST].values
)

Simulation: SUCCESS
Pressure at test junction: [32.347 31.76  26.063 27.468 27.363 27.079 26.539 25.228 24.175 23.831 30.739 31.043 31.354 31.704 32.001 32.259 32.448 32.515 32.459 32.303 32.32  54.53  56.836 58.359 59.073]


### 4. Reservoir head magnitude

You added another reservoir previously, but did not systematically vary its operating head.

In [12]:
wn = copy.deepcopy(base_wn)

reservoir = wn.get_node(R_TEST)
original_head = reservoir.head_timeseries.base_value

for factor in [0.90, 1.00, 1.10]:

    wn = copy.deepcopy(base_wn)
    reservoir = wn.get_node(R_TEST)

    reservoir.head_timeseries.base_value = original_head * factor

    print("\nFactor:", factor)
    print("New reservoir head:", reservoir.head_timeseries.base_value)

    assert np.isclose(
        reservoir.head_timeseries.base_value,
        original_head * factor
    )

    results = run_hydraulics(wn)

    print("Verified: YES")
    print("Mean network pressure:",
          results.node["pressure"].iloc[0].mean())


Factor: 0.9
New reservoir head: 139.757179464
Verified: YES
Mean network pressure: 63.010044

Factor: 1.0
New reservoir head: 155.28575496
Verified: YES
Mean network pressure: 63.345814

Factor: 1.1
New reservoir head: 170.814330456
Verified: YES
Mean network pressure: 63.694695


Reservoir head is stored as a time series, so its base value can be modified directly. [EPA GitHub Pages](https://usepa.github.io/WNTR/waternetworkmodel.html)

### 5. Reservoir head pattern

Now make the source head vary through time.

In [13]:
wn = copy.deepcopy(base_wn)

head_pattern = np.array([
    1.00, 1.00, 1.00, 0.99, 0.99, 0.98,
    0.98, 0.99, 1.00, 1.01, 1.02, 1.02,
    1.02, 1.01, 1.00, 0.99, 0.98, 0.98,
    0.99, 1.00, 1.01, 1.01, 1.00, 1.00
])

head_pattern = head_pattern / head_pattern.mean()

pattern_name = "Experimental_24h_Reservoir"

wn.add_pattern(pattern_name, head_pattern)

reservoir = wn.get_node(R_TEST)
reservoir.head_timeseries.pattern_name = pattern_name

print("Reservoir:", R_TEST)
print("Pattern:", reservoir.head_timeseries.pattern_name)
print("Average multiplier:", head_pattern.mean())

assert reservoir.head_timeseries.pattern_name == pattern_name

print("Verified: YES")

Reservoir: R-1
Pattern: Experimental_24h_Reservoir
Average multiplier: 1.0
Verified: YES


In [14]:
results = run_hydraulics(wn, hours=24)

print("Simulation: SUCCESS")
print("Reservoir-head-driven pressure range:")
print(
    results.node["pressure"][J_TEST].min(),
    "to",
    results.node["pressure"][J_TEST].max()
)

Simulation: SUCCESS
Reservoir-head-driven pressure range:
24.286549 to 59.246162


### 6. Tank initial level

You already created a tank; now vary its **initial state**

In [15]:
wn = copy.deepcopy(base_wn)

tank = wn.get_node(T_TEST)

min_level = tank.min_level
max_level = tank.max_level

for fraction in [0.20, 0.50, 0.80]:

    wn = copy.deepcopy(base_wn)
    tank = wn.get_node(T_TEST)

    new_level = min_level + fraction * (max_level - min_level)
    tank.init_level = new_level

    print("\nInitial level:", tank.init_level)

    assert np.isclose(tank.init_level, new_level)

    results = run_hydraulics(wn, hours=24)

    print("Verified: YES")
    print(
        "Final tank level:",
        results.node["head"][T_TEST].iloc[-1] - tank.elevation
    )


Initial level: 27.766454601600003
Verified: YES
Final tank level: 36.300858

Initial level: 30.966855516000003
Verified: YES
Final tank level: 36.300858

Initial level: 34.1672564304
Verified: YES
Final tank level: 36.300858


Tank `init_level`, `min_level`, `max_level`, etc. are directly modifiable attributes. [EPA GitHub Pages](https://usepa.github.io/WNTR/apidoc/wntr.network.elements.Tank.html)

### 7. Pipe roughness — systematic variation

You already changed roughness once. Now test it systematically

In [16]:
wn = copy.deepcopy(base_wn)

pipe = wn.get_link(P_TEST)
original_roughness = pipe.roughness

for factor in [0.80, 1.00, 1.20]:

    wn = copy.deepcopy(base_wn)
    pipe = wn.get_link(P_TEST)

    pipe.roughness = original_roughness * factor

    print("\nRoughness:", pipe.roughness)

    assert np.isclose(
        pipe.roughness,
        original_roughness * factor
    )

    results = run_hydraulics(wn)

    print("Verified: YES")
    print("Mean pressure:",
          results.node["pressure"].iloc[0].mean())


Roughness: 120.0
Verified: YES
Mean pressure: 63.345814

Roughness: 150.0
Verified: YES
Mean pressure: 63.345814

Roughness: 180.0
Verified: YES
Mean pressure: 63.345814


### 8. Pipe minor-loss coefficient

This has not been tested yet

In [17]:
wn = copy.deepcopy(base_wn)

pipe = wn.get_link(P_TEST)

for K in [0.0, 1.0, 5.0, 10.0]:

    wn = copy.deepcopy(base_wn)
    pipe = wn.get_link(P_TEST)

    pipe.minor_loss = K

    print("\nMinor-loss coefficient:", pipe.minor_loss)

    assert np.isclose(pipe.minor_loss, K)

    results = run_hydraulics(wn)

    print("Verified: YES")
    print("Mean pressure:",
          results.node["pressure"].iloc[0].mean())


Minor-loss coefficient: 0.0
Verified: YES
Mean pressure: 63.345814

Minor-loss coefficient: 1.0
Verified: YES
Mean pressure: 63.345818

Minor-loss coefficient: 5.0
Verified: YES
Mean pressure: 63.345814

Minor-loss coefficient: 10.0
Verified: YES
Mean pressure: 63.345814


### 9. Pump speed

This is genuinely new relative to your notebook

In [19]:
if PU_TEST is not None:

    wn = copy.deepcopy(base_wn)

    pump = wn.get_link(PU_TEST)
    original_speed = pump.base_speed

    for factor in [0.90, 1.00, 1.10]:

        wn = copy.deepcopy(base_wn)
        pump = wn.get_link(PU_TEST)

        pump.base_speed = original_speed * factor

        print("\nPump:", PU_TEST)
        print("Speed:", pump.base_speed)

        assert np.isclose(
            pump.base_speed,
            original_speed * factor
        )

        results = run_hydraulics(wn)

        print("Verified: YES")
        print(
            "Pump flowrate:",
            results.link["flowrate"][PU_TEST].iloc[0]
        )


Pump: ~@Pump-1
Speed: 0.9
Verified: YES
Pump flowrate: 0.0

Pump: ~@Pump-1
Speed: 1.0
Verified: YES
Pump flowrate: 0.0

Pump: ~@Pump-1
Speed: 1.1
Verified: YES
Pump flowrate: 0.0


Pump speed is represented by a base speed/time series in WNTR. [EPA GitHub Pages](https://usepa.github.io/WNTR/apidoc/wntr.network.elements.Pump.html)

### 10. Pump ON/OFF status

You have already changed statuses for other links, but not tested the **hydraulic effect of a pump outage**

In [20]:
if PU_TEST is not None:

    wn = copy.deepcopy(base_wn)

    pump = wn.get_link(PU_TEST)

    print("Original status:", pump.initial_status)

    pump.initial_status = "CLOSED"

    print("New status:", pump.initial_status)

    assert str(pump.initial_status).upper().endswith("CLOSED")

    try:
        results = run_hydraulics(wn)

        print("Verified: status changed")
        print("Hydraulic simulation: SUCCESS")

    except Exception as e:
        print("Status change: VERIFIED")
        print("Hydraulic simulation failed:")
        print(type(e).__name__, e)
        print("This may mean this pump is hydraulically critical.")

Original status: Open
New status: Closed
Verified: status changed
Hydraulic simulation: SUCCESS


Pump initial status can be `"OPEN"` or `"CLOSED"`. [EPA GitHub Pages](https://usepa.github.io/WNTR/apidoc/wntr.network.model.LinkRegistry.html)

### 11. Valve setting

Because valve settings mean different things for different valve types, we should **inspect the valve type first**

In [21]:
if V_TEST is not None:

    valve = base_wn.get_link(V_TEST)

    print("Valve:", V_TEST)
    print("Type :", valve.valve_type)
    print("Initial setting:", valve.initial_setting)

Valve: ~@RV-1
Type : PRV
Initial setting: 91.44000000000001


For a **PRV/PSV/PBV**, the setting is a pressure setting.  
For an **FCV**, it is a flow setting. [EPA GitHub Pages](https://usepa.github.io/WNTR/apidoc/wntr.network.model.LinkRegistry.html)

##### Automatic experiment for PRV/FCV

In [22]:
if V_TEST is not None:

    wn = copy.deepcopy(base_wn)
    valve = wn.get_link(V_TEST)

    if valve.valve_type in {"PRV", "PSV", "PBV"}:

        upstream_pressure = baseline_results.node[
            "pressure"
        ][valve.start_node_name].iloc[0]

        if valve.initial_setting > 0:
            settings = [
                0.80 * valve.initial_setting,
                1.00 * valve.initial_setting,
                1.20 * valve.initial_setting
            ]
        elif upstream_pressure > 0:
            settings = [
                0.50 * upstream_pressure,
                0.70 * upstream_pressure,
                0.90 * upstream_pressure
            ]
        else:
            settings = []

    elif valve.valve_type == "FCV":

        baseline_flow = abs(
            baseline_results.link["flow"][V_TEST].iloc[0]
        )

        settings = [
            0.80 * baseline_flow,
            1.00 * baseline_flow,
            1.20 * baseline_flow
        ]

    else:
        settings = []

        print(
            "Automatic sweep skipped for valve type:",
            valve.valve_type
        )

    for setting in settings:

        wn = copy.deepcopy(base_wn)
        valve = wn.get_link(V_TEST)

        valve.initial_setting = setting

        print("\nValve:", V_TEST)
        print("Type:", valve.valve_type)
        print("Setting:", valve.initial_setting)

        assert np.isclose(
            valve.initial_setting,
            setting
        )

        results = run_hydraulics(wn)

        print("Verified: YES")


Valve: ~@RV-1
Type: PRV
Setting: 73.15200000000002
Verified: YES

Valve: ~@RV-1
Type: PRV
Setting: 91.44000000000001
Verified: YES

Valve: ~@RV-1
Type: PRV
Setting: 109.72800000000001
Verified: YES


### 12. Leak location — systematic spatial variation

You already demonstrated changing emitter location manually. Now make **location itself a scenario variable** across the realistic network.

First select candidate junctions having positive baseline pressure and non-zero demand.

In [ ]:
pressure0 = baseline_results.node["pressure"].iloc[0]

candidate_nodes = []

for name in base_wn.junction_name_list:

    node = base_wn.get_node(name)

    demand = sum(
        ts.base_value
        for ts in node.demand_timeseries_list
    )

    if pressure0[name] > 0 and demand > 0:
        candidate_nodes.append(name)

print("Candidate leak nodes:", len(candidate_nodes))

Candidate leak nodes: 1030


Select several reproducibly:

In [24]:
rng = np.random.default_rng(42)

LEAK_NODES = rng.choice(
    candidate_nodes,
    size=min(5, len(candidate_nodes)),
    replace=False
).tolist()

print("Selected leak locations:")
print(LEAK_NODES)

Selected leak locations:
['J-747', 'J-392', 'J-614', 'J-1089', 'J-385']


Now run each location.

In [25]:
for leak_node in LEAK_NODES:

    wn = copy.deepcopy(base_wn)

    # Ensure there is only ONE leak
    for name in wn.junction_name_list:
        wn.get_node(name).emitter_coefficient = 0.0

    node = wn.get_node(leak_node)

    demand = sum(
        ts.base_value
        for ts in node.demand_timeseries_list
    )

    pressure = baseline_results.node[
        "pressure"
    ][leak_node].iloc[0]

    exponent = wn.options.hydraulic.emitter_exponent

    # Leak target = 10% of normal local demand
    target_leak_flow = 0.10 * demand

    emitter_C = target_leak_flow / (pressure ** exponent)

    node.emitter_coefficient = emitter_C

    print("\nLeak node:", leak_node)
    print("Emitter coefficient:", node.emitter_coefficient)

    assert np.isclose(
        node.emitter_coefficient,
        emitter_C
    )

    results = run_hydraulics(wn)

    print("Verified: YES")
    print(
        "Leak-node pressure:",
        results.node["pressure"][leak_node].iloc[0]
    )



Leak node: J-747
Emitter coefficient: 2.5317718e-07
Verified: YES
Leak-node pressure: 54.989788

Leak node: J-392
Emitter coefficient: 4.0977832e-07
Verified: YES
Leak-node pressure: 83.94794

Leak node: J-614
Emitter coefficient: 8.571528e-07
Verified: YES
Leak-node pressure: 43.17351

Leak node: J-1089
Emitter coefficient: 8.001372e-08
Verified: YES
Leak-node pressure: 61.159607

Leak node: J-385
Emitter coefficient: 3.8155764e-07
Verified: YES
Leak-node pressure: 67.24316


This makes the leak severity interpretable: the coefficient is calculated so that, under the baseline pressure, the leak would approximately represent **10% of that junction's normal demand**.

### 13. Leak severity

Now keep the **same location** and change only the leak magnitude.

In [26]:
LEAK_NODE = LEAK_NODES[0]

wn = copy.deepcopy(base_wn)

node = wn.get_node(LEAK_NODE)

demand = sum(
    ts.base_value
    for ts in node.demand_timeseries_list
)

pressure = baseline_results.node[
    "pressure"
][LEAK_NODE].iloc[0]

exponent = wn.options.hydraulic.emitter_exponent

for leak_fraction in [0.05, 0.10, 0.20, 0.30]:

    wn = copy.deepcopy(base_wn)

    # Remove all previous leaks
    for name in wn.junction_name_list:
        wn.get_node(name).emitter_coefficient = 0.0

    node = wn.get_node(LEAK_NODE)

    target_leak_flow = leak_fraction * demand

    emitter_C = target_leak_flow / (pressure ** exponent)

    node.emitter_coefficient = emitter_C

    print("\nLeak fraction:", leak_fraction)
    print("Emitter coefficient:", emitter_C)

    assert np.isclose(
        node.emitter_coefficient,
        emitter_C
    )

    results = run_hydraulics(wn)

    print("Verified: YES")
    print(
        "Leak-node pressure:",
        results.node["pressure"][LEAK_NODE].iloc[0]
    )


Leak fraction: 0.05
Emitter coefficient: 1.2658859e-07
Verified: YES
Leak-node pressure: 54.989788

Leak fraction: 0.1
Emitter coefficient: 2.5317718e-07
Verified: YES
Leak-node pressure: 54.989788

Leak fraction: 0.2
Emitter coefficient: 5.0635435e-07
Verified: YES
Leak-node pressure: 54.989788

Leak fraction: 0.3
Emitter coefficient: 7.595315e-07
Verified: YES
Leak-node pressure: 54.989788


### 14. Multiple simultaneous leaks

The toy notebook technically had two emitters, so let's not treat "two emitters exist" as new. The new part here is a **controlled multiple-leak scenario**.

In [27]:
MULTI_LEAK_NODES = LEAK_NODES[:3]

wn = copy.deepcopy(base_wn)

# Remove all existing emitters
for name in wn.junction_name_list:
    wn.get_node(name).emitter_coefficient = 0.0

for leak_node in MULTI_LEAK_NODES:

    node = wn.get_node(leak_node)

    demand = sum(
        ts.base_value
        for ts in node.demand_timeseries_list
    )

    pressure = baseline_results.node[
        "pressure"
    ][leak_node].iloc[0]

    exponent = wn.options.hydraulic.emitter_exponent

    target_leak_flow = 0.10 * demand

    emitter_C = target_leak_flow / (pressure ** exponent)

    node.emitter_coefficient = emitter_C

print("Active leak nodes:")

for name in wn.junction_name_list:
    C = wn.get_node(name).emitter_coefficient

    if C > 0:
        print(name, "->", C)

assert sum(
    wn.get_node(name).emitter_coefficient > 0
    for name in wn.junction_name_list
) == len(MULTI_LEAK_NODES)

print("Verified: YES")

Active leak nodes:
J-392 -> 4.0977832e-07
J-614 -> 8.571528e-07
J-747 -> 2.5317718e-07
Verified: YES


In [28]:
results = run_hydraulics(wn)

print("Multiple-leak simulation: SUCCESS")

for leak_node in MULTI_LEAK_NODES:
    print(
        leak_node,
        "pressure =",
        results.node["pressure"][leak_node].iloc[0]
    )


Multiple-leak simulation: SUCCESS
J-747 pressure = 54.989807
J-392 pressure = 83.95868
J-614 pressure = 43.174553
